In [33]:
STUDENT_FILE = "students.txt"
USER_FILE = "users.txt"
REPORT_FILE = "report.txt"
SEP = "|"       

In [47]:
def load_students(path):
    """Read students from a text file into a dictionary and return it."""
    records = {}
    f = None
    try:
        f = open(path, "r")
        for line in f:                     
            line = line.strip()
            if line == "":
                continue                  
            parts = line.split(SEP)         
            if len(parts) != 4:
                print("Skipping bad line:", line)
                continue
            try:
                marks = float(parts[3])
            except ValueError:
                print("Skipping line with invalid marks:", line)
                continue
            records[parts[0]] = {"name": parts[1], "course": parts[2], "marks": marks}
    except FileNotFoundError:
        print("File not found. A new one will be created when you save.")
    except PermissionError:
        print("No permission to read", path)
    finally:
        if f:
            f.close()                      
    return records


def load_users(path):
    """Read 'username|password' lines into a dictionary."""
    users = {}
    try:
        with open(path, "r") as f:
            for line in f:
                parts = line.strip().split(SEP)
                if len(parts) == 2:
                    users[parts[0]] = parts[1]
    except FileNotFoundError:
        print("Users file missing. Creating default admin account.")
        with open(path, "w") as f:
            f.write("admin" + SEP + "admin123\n")
        users["admin"] = "admin123"
    except PermissionError:
        print("No permission to read", path)
    return users



In [35]:
def save_students(path, records):
    """Overwrite the file with all current records. Returns True/False."""
    try:
        f = open(path, "w")                 # "w" = write (erases old content)
        for sid, d in records.items():
            f.write(sid + SEP + d["name"] + SEP + d["course"] + SEP + str(d["marks"]) + "\n")
        f.close()
        return True
    except PermissionError:
        print("No permission to write", path)
        return False


def append_student(path, sid, d):
    """Add one new line to the end of the file without erasing the rest."""
    try:
        with open(path, "a") as f:          # "a" = append
            f.write(sid + SEP + d["name"] + SEP + d["course"] + SEP + str(d["marks"]) + "\n")
        return True
    except PermissionError:
        print("No permission to write", path)
        return False


Username:  ee
Password:  ee


Wrong login. Attempts left: 2


Username:  eee
Password:  eee


Wrong login. Attempts left: 1


Username:  ee
Password:  eee


Wrong login. Attempts left: 0
Too many wrong attempts.


In [36]:
def user_login(users, username, password):
    """Return True if username and password match, else False."""
    if username in users and users[username] == password:
        return True
    return False


def login_screen(users):
    """Give the user 3 attempts. Returns True if login succeeds."""
    attempts = 0
    while attempts < 3:
        username = input("Username: ").strip()
        password = input("Password: ").strip()
        if user_login(users, username, password):
            print("Login successful. Welcome,", username)
            return True
        attempts += 1
        print("Wrong login. Attempts left:", 3 - attempts)
    print("Too many wrong attempts.")
    return False

In [37]:
def add_record(records, sid, name, course, marks):
    """Add a student to the dictionary. Returns (True/False, message)."""
    if sid in records:
        return False, "ID already exists."
    records[sid] = {"name": name, "course": course, "marks": marks}
    return True, "Student added."


def search_record(records, keyword):
    """Return a list of students whose ID or name matches the keyword."""
    keyword = keyword.lower()
    results = []
    for sid, d in records.items():
        if keyword == sid.lower() or keyword in d["name"].lower():
            results.append((sid, d))
    return results


def update_record(records, sid, marks):
    """Change the marks of an existing student."""
    if sid not in records:
        return False, "Student not found."
    records[sid]["marks"] = marks
    return True, "Marks updated."


def delete_record(records, sid):
    """Remove a student by ID."""
    if sid not in records:
        return False, "Student not found."
    del records[sid]
    return True, "Record deleted."

In [38]:
def rank_students(records):
    """Return a list of (id, data) sorted from highest marks to lowest."""
    items = list(records.items())           # turn dictionary into a list
    n = len(items)
    for i in range(n):                      # simple bubble sort
        for j in range(n - i - 1):
            if items[j][1]["marks"] < items[j + 1][1]["marks"]:
                items[j], items[j + 1] = items[j + 1], items[j]   # swap
    return items

In [39]:
def get_grade(marks):
    if marks >= 85:
        return "A"
    elif marks >= 70:
        return "B"
    elif marks >= 55:
        return "C"
    elif marks >= 40:
        return "D"
    else:
        return "F"


def generate_report(records):
    """Return a text report with average, highest, lowest and grade counts."""
    if len(records) == 0:
        return "No records available."

    total = 0
    highest_name, highest_marks = "", -1
    lowest_name, lowest_marks = "", 101
    grades = {"A": 0, "B": 0, "C": 0, "D": 0, "F": 0}

    for sid, d in records.items():
        m = d["marks"]
        total = total + m
        grades[get_grade(m)] += 1
        if m > highest_marks:
            highest_name, highest_marks = d["name"], m
        if m < lowest_marks:
            lowest_name, lowest_marks = d["name"], m

    report = "===== STUDENT REPORT =====\n"
    report += "Total students : " + str(len(records)) + "\n"
    report += "Average marks  : " + str(round(total / len(records), 2)) + "\n"
    report += "Highest        : " + highest_name + " (" + str(highest_marks) + ")\n"
    report += "Lowest         : " + lowest_name + " (" + str(lowest_marks) + ")\n"
    report += "Grades         : "
    for g, count in grades.items():
        report += g + "=" + str(count) + " "
    return report


def save_report(text):
    """Append the report to report.txt."""
    try:
        with open(REPORT_FILE, "a") as f:
            f.write(text + "\n\n")
        print("Report saved to", REPORT_FILE)
    except PermissionError:
        print("Cannot save report: no permission.")

In [40]:
def get_valid_marks():
    """Keep asking until the user types a number from 0 to 100."""
    while True:
        try:
            marks = float(input("Enter marks (0-100): "))
            if marks >= 0 and marks <= 100:
                return marks
            print("Marks must be between 0 and 100.")
        except ValueError:
            print("Invalid input. Please type a number.")

In [41]:
def display_records(records):
    if len(records) == 0:
        print("No records found.")
        return
    print("ID      Name                Course         Marks")
    for sid, d in records.items():
        print(sid.ljust(8) + d["name"].ljust(20) + d["course"].ljust(15) + str(d["marks"]))


def main():
    users = load_users(USER_FILE)
    records = load_students(STUDENT_FILE)      

    if not login_screen(users):
        return

    while True:                                 
        print("\n--- STUDENT PORTAL ---")
        print("1. Add student")
        print("2. Search student")
        print("3. Update marks")
        print("4. Delete student")
        print("5. View all students")
        print("6. Generate report")
        print("7. Exit")
        print("8. Show ranking")
        choice = input("Choose (1-8): ").strip()

        if choice == "1":
            sid = input("ID: ").strip()
            name = input("Name: ").strip()
            course = input("Course: ").strip()
            if sid == "" or name == "" or course == "":
                print("All fields are required.")
                continue
            if SEP in sid + name + course:
                print("The character", SEP, "is not allowed.")
                continue
            marks = get_valid_marks()
            ok, msg = add_record(records, sid, name, course, marks)
            print(msg)
            if ok:
                append_student(STUDENT_FILE, sid, records[sid])

        elif choice == "2":
            found = search_record(records, input("Enter ID or name: ").strip())
            for sid, d in found:
                print(sid, d["name"], d["course"], d["marks"])
            if len(found) == 0:
                print("No match found.")

        elif choice == "3":
            sid = input("ID: ").strip()
            if sid in records:
                ok, msg = update_record(records, sid, get_valid_marks())
                print(msg)
                save_students(STUDENT_FILE, records)
            else:
                print("Student not found.")

        elif choice == "4":
            ok, msg = delete_record(records, input("ID: ").strip())
            print(msg)
            if ok:
                save_students(STUDENT_FILE, records)

        elif choice == "5":
            display_records(records)

        elif choice == "6":
            report = generate_report(records)
            print(report)
            save_report(report)

        elif choice == "7":
            print("Goodbye!")
            break

        elif choice == "8":
            ranked = rank_students(records)
            position = 1
            for sid, d in ranked:
                print(position, d["name"], d["marks"])
                position += 1

        else:
            print("Invalid choice. Please enter a number from 1 to 8.")

In [48]:
test = {}
print(add_record(test, "S1", "Ayesha", "PAI", 88.5))
print(add_record(test, "S2", "Bilal", "PAI", 62))
print(add_record(test, "S3", "Hamza", "PAI", 95))
print(add_record(test, "S1", "Duplicate", "PAI", 50))  
print(search_record(test, "ayesha"))
print(update_record(test, "S2", 75))
print(delete_record(test, "S9"))                          
print()
print(generate_report(test))
print()
print("Ranking (highest first):")
position = 1
for sid, d in rank_students(test):
    print(position, d["name"], d["marks"])
    position += 1
print()
save_students("test_students.txt", test)                  
print("Read back from file:")
print(load_students("test_students.txt"))

(True, 'Student added.')
(True, 'Student added.')
(True, 'Student added.')
(False, 'ID already exists.')
[('S1', {'name': 'Ayesha', 'course': 'PAI', 'marks': 88.5})]
(True, 'Marks updated.')
(False, 'Student not found.')

===== STUDENT REPORT =====
Total students : 3
Average marks  : 86.17
Highest        : Hamza (95)
Lowest         : Bilal (75)
Grades         : A=2 B=1 C=0 D=0 F=0 

Ranking (highest first):
1 Hamza 95
2 Ayesha 88.5
3 Bilal 75

Read back from file:
{'S1': {'name': 'Ayesha', 'course': 'PAI', 'marks': 88.5}, 'S2': {'name': 'Bilal', 'course': 'PAI', 'marks': 75.0}, 'S3': {'name': 'Hamza', 'course': 'PAI', 'marks': 95.0}}


In [49]:
print("Missing file:")
print(load_students("does_not_exist.txt"))

print("\nCorrupted file:")
f = open("bad_students.txt", "w")
f.write("S1|Ayesha|PAI|88\n")
f.write("S2|Bilal|PAI|abc\n")       
f.write("S3|OnlyTwoFields\n")         
f.close()
print(load_students("bad_students.txt"))

Missing file:
File not found. A new one will be created when you save.
{}

Corrupted file:
Skipping line with invalid marks: S2|Bilal|PAI|abc
Skipping bad line: S3|OnlyTwoFields
{'S1': {'name': 'Ayesha', 'course': 'PAI', 'marks': 88.0}}


In [50]:
try:
    f = open(STUDENT_FILE, "r")
    f.close()
    print(STUDENT_FILE, "already exists. Nothing changed.")
except FileNotFoundError:
    append_student(STUDENT_FILE, "S1", {"name": "Ayesha", "course": "PAI", "marks": 88.5})
    append_student(STUDENT_FILE, "S2", {"name": "Bilal", "course": "PAI", "marks": 62})
    append_student(STUDENT_FILE, "S3", {"name": "Hamza", "course": "PAI", "marks": 95})
    print(STUDENT_FILE, "created with 3 sample students.")

students.txt already exists. Nothing changed.


In [ ]:
main()

Username:  admin
Password:  admin123


Login successful. Welcome, admin

--- STUDENT PORTAL ---
1. Add student
2. Search student
3. Update marks
4. Delete student
5. View all students
6. Generate report
7. Exit
8. Show ranking


Choose (1-8):  6


===== STUDENT REPORT =====
Total students : 3
Average marks  : 81.83
Highest        : Hamza (95.0)
Lowest         : Bilal (62.0)
Grades         : A=2 B=0 C=1 D=0 F=0 
Report saved to report.txt

--- STUDENT PORTAL ---
1. Add student
2. Search student
3. Update marks
4. Delete student
5. View all students
6. Generate report
7. Exit
8. Show ranking


Choose (1-8):  4
ID:  8


Student not found.

--- STUDENT PORTAL ---
1. Add student
2. Search student
3. Update marks
4. Delete student
5. View all students
6. Generate report
7. Exit
8. Show ranking


... (6 outputs left out)

Choose (1-8):  3
ID:  S2
Enter marks (0-100):  66


Marks updated.

--- STUDENT PORTAL ---
1. Add student
2. Search student
3. Update marks
4. Delete student
5. View all students
6. Generate report
7. Exit
8. Show ranking


Choose (1-8):  2
Enter ID or name:  bilal


S2 Bilal PAI 66.0

--- STUDENT PORTAL ---
1. Add student
2. Search student
3. Update marks
4. Delete student
5. View all students
6. Generate report
7. Exit
8. Show ranking
